# Road segmentation: test on a video recorded by the robot

Runs the trained model frame by frame on a video recorded with the TurtleBot3 camera and displays the segmentation masks (road, pedestrian crossings, arrows, robot, parking…).

## 1. Environment and model

In [ ]:
!pip install ultralytics

#import yolov8 libraries
#https://docs.ultralytics.com/

import ultralytics
from ultralytics import YOLO

import cv2
import numpy as np
from google.colab.patches import cv2_imshow

# mount google colab
from google.colab import drive
drive.mount('/content/drive')

#on charge notre modèle, entrainé sur notre dataset
#tutorial : https://docs.ultralytics.com/modes/predict/#inference-sources

model = YOLO('/content/drive/MyDrive/road_tracking/best.onnx',task='segment') # your trained model

## 2. Segmentation on the video

In [ ]:
#on ouvre un fichier video
cap = cv2.VideoCapture('/content/drive/MyDrive/road_tracking/video_course3.mp4') #à modifier selon vos conditions

#read the video file until "q" is pressed
while( cap.isOpened() ):
    ret, frame = cap.read()
    if not ret:
        break

    # frame_resized = cv2.resize(frame, (128, 128)) # add resize command to fit model size
    results = model.predict(frame,imgsz=[128,128],conf=0.15)  # predict on an image

    if len(results) > 0:
        for r in results:
            res_plotted = r.plot() # get results
            box = r.boxes.cpu() #get bounding boxes
            box_sz = box.xyxy.numpy() # coordinates
            if box_sz.size !=0 : # if at least one object is detected
            #            cv2.circle( res_plotted, (x, y), 4, (0, 0, 255), -1 ) utilisez ces fonctions pour déssiner les cirles
            #            cv2.line(res_plotted,(x,y),(xim,yim),(255,0,0),2)
              cv2_imshow(res_plotted)
            break

    # c = cv2.waitKey(1) # Removed as it requires a GUI
    # if c == 27:
    #     break

cap.release()
cv2.destroyAllWindows()

## 3. Re-export the PyTorch weights to ONNX opset 11

In [ ]:
# First, make sure you have your original trained PyTorch model file.
# For example, if your PyTorch model is named 'best.pt'

# Load the PyTorch model
model_pt = YOLO('/content/drive/MyDrive/road_tracking/best.pt') # Replace with the actual path to your .pt model

# Export the model to ONNX with opset=11
# You might need to adjust the name if you want to keep the old one
model_pt.export(format='onnx', opset=11, simplify=True, name='best_opset11.onnx')

print("Model re-exported successfully to 'best_opset11.onnx' with opset 11.")